# 05 - Test Recommendation Engine

This notebook tests the SafePlate recommendation engine that uses Node2Vec embeddings
to find similar ingredients based on their co-occurrence patterns in recipes.

**Date:** February 2026  
**Author:** SafePlate Team

## Setup

In [1]:
import sys
from pathlib import Path

# Add backend to path so we can import safeplate modules
project_root = Path.cwd().parent
backend_path = project_root / "backend"
sys.path.insert(0, str(backend_path))

print(f"Project root: {project_root}")
print(f"Backend path: {backend_path}")

Project root: /Users/adarpalaoglu/Documents/cs310-safeplate
Backend path: /Users/adarpalaoglu/Documents/cs310-safeplate/backend


In [2]:
# Import our modules
from safeplate.embedding_loader import EmbeddingLoader, get_embedding_loader
from safeplate.recommendation_engine import RecommendationEngine, IngredientNotFoundError
from safeplate.utils import cosine_similarity, normalize_ingredient_name

import numpy as np
import pandas as pd

## Test 1: Embedding Loader

In [3]:
# Initialize the embedding loader
embedding_path = project_root / "data" / "processed" / "embeddings" / "ingredient_node2vec_iter10_UNWEIGHTED_p1_q4_w10_wl80_r80_e10.txt"
loader = EmbeddingLoader(str(embedding_path))

# Load embeddings
loader.load()

print(f"Number of ingredients: {loader.num_ingredients}")
print(f"Embedding dimension: {loader.embedding_dim}")

Number of ingredients: 2698
Embedding dimension: 128


In [4]:
# Test getting individual embeddings
test_ingredients = ["tomato", "chicken", "milk", "garlic clove"]

for ing in test_ingredients:
    emb = loader.get_embedding(ing)
    if emb is not None:
        print(f"{ing}: shape={emb.shape}, norm={np.linalg.norm(emb):.4f}")
    else:
        print(f"{ing}: NOT FOUND")

tomato: shape=(128,), norm=1.0491
chicken: shape=(128,), norm=1.0661
milk: shape=(128,), norm=1.2597
garlic clove: shape=(128,), norm=1.0130


In [5]:
# Show first 20 ingredients
all_ingredients = loader.get_all_ingredients()
print(f"First 20 ingredients:")
for i, ing in enumerate(all_ingredients[:20]):
    print(f"  {i+1}. {ing}")

First 20 ingredients:
  1. egg
  2. flour
  3. garlic clove
  4. milk
  5. tomato
  6. chicken
  7. cream
  8. garlic
  9. parsley
  10. cinnamon
  11. lemon juice
  12. red pepper
  13. carrot
  14. powder
  15. parmesan cheese
  16. soda
  17. ginger
  18. basil
  19. cumin
  20. oregano


## Test 2: Recommendation Engine - Basic Queries

In [6]:
# Initialize recommendation engine
engine = RecommendationEngine(loader)

In [7]:
# Query 1: Similar to "tomato"
print("=" * 50)
print("Similar ingredients to 'tomato':")
print("=" * 50)

tomato_similar = engine.get_similar_ingredients("tomato", top_n=10)
for i, (ing, score) in enumerate(tomato_similar, 1):
    print(f"  {i:2}. {ing:<30} (similarity: {score:.4f})")

Similar ingredients to 'tomato':
   1. garlic clove                   (similarity: 0.8474)
   2. oregano                        (similarity: 0.8359)
   3. garlic                         (similarity: 0.8342)
   4. red pepper                     (similarity: 0.8320)
   5. plum tomato                    (similarity: 0.8126)
   6. parsley                        (similarity: 0.8024)
   7. salt pepper                    (similarity: 0.7827)
   8. cayenne pepper                 (similarity: 0.7812)
   9. tomato sauce                   (similarity: 0.7782)
  10. cumin                          (similarity: 0.7749)


In [8]:
# Query 2: Similar to "chicken"
print("=" * 50)
print("Similar ingredients to 'chicken':")
print("=" * 50)

chicken_similar = engine.get_similar_ingredients("chicken", top_n=10)
for i, (ing, score) in enumerate(chicken_similar, 1):
    print(f"  {i:2}. {ing:<30} (similarity: {score:.4f})")

Similar ingredients to 'chicken':
   1. chicken broth                  (similarity: 0.8146)
   2. garlic                         (similarity: 0.8064)
   3. red pepper                     (similarity: 0.8052)
   4. chicken thigh                  (similarity: 0.7994)
   5. garlic clove                   (similarity: 0.7894)
   6. parsley                        (similarity: 0.7330)
   7. tomato                         (similarity: 0.7310)
   8. cayenne pepper                 (similarity: 0.7309)
   9. rice                           (similarity: 0.7286)
  10. mushroom                       (similarity: 0.7140)


In [9]:
# Query 3: Similar to "milk"
print("=" * 50)
print("Similar ingredients to 'milk':")
print("=" * 50)

milk_similar = engine.get_similar_ingredients("milk", top_n=10)
for i, (ing, score) in enumerate(milk_similar, 1):
    print(f"  {i:2}. {ing:<30} (similarity: {score:.4f})")

Similar ingredients to 'milk':
   1. egg                            (similarity: 0.8934)
   2. flour                          (similarity: 0.8645)
   3. margarine                      (similarity: 0.8567)
   4. cream                          (similarity: 0.8131)
   5. powder                         (similarity: 0.7876)
   6. shortening                     (similarity: 0.7826)
   7. 2 milk                         (similarity: 0.7812)
   8. vanilla extract                (similarity: 0.7777)
   9. buttermilk                     (similarity: 0.7713)
  10. soda                           (similarity: 0.7644)


## Test 3: Filtering (Exclude List)

In [10]:
# Find similar to milk, but exclude dairy-related ingredients
dairy_triggers = ["cream", "butter", "cheese", "yogurt", "sour cream"]

print("=" * 50)
print("Similar to 'milk' (excluding dairy triggers):")
print(f"Excluded: {dairy_triggers}")
print("=" * 50)

milk_filtered = engine.get_similar_ingredients(
    "milk", 
    top_n=10, 
    exclude_list=dairy_triggers
)

for i, (ing, score) in enumerate(milk_filtered, 1):
    print(f"  {i:2}. {ing:<30} (similarity: {score:.4f})")

Similar to 'milk' (excluding dairy triggers):
Excluded: ['cream', 'butter', 'cheese', 'yogurt', 'sour cream']
   1. egg                            (similarity: 0.8934)
   2. flour                          (similarity: 0.8645)
   3. margarine                      (similarity: 0.8567)
   4. powder                         (similarity: 0.7876)
   5. shortening                     (similarity: 0.7826)
   6. 2 milk                         (similarity: 0.7812)
   7. vanilla extract                (similarity: 0.7777)
   8. buttermilk                     (similarity: 0.7713)
   9. soda                           (similarity: 0.7644)
  10. cream cheese                   (similarity: 0.7619)


In [11]:
# Use the find_substitutes method for dietary triggers
print("=" * 50)
print("Finding substitutes for 'egg' (avoiding common allergens):")
print("=" * 50)

allergens = ["milk", "cream", "butter", "peanut", "tree nut"]
egg_substitutes = engine.find_substitutes("egg", triggers=allergens, top_n=10)

for i, (ing, score) in enumerate(egg_substitutes, 1):
    print(f"  {i:2}. {ing:<30} (similarity: {score:.4f})")

Finding substitutes for 'egg' (avoiding common allergens):
   1. flour                          (similarity: 0.9127)
   2. powder                         (similarity: 0.8795)
   3. margarine                      (similarity: 0.8537)
   4. soda                           (similarity: 0.8477)
   5. vanilla extract                (similarity: 0.8387)
   6. vanilla                        (similarity: 0.8357)
   7. shortening                     (similarity: 0.8350)
   8. buttermilk                     (similarity: 0.8320)
   9. cinnamon                       (similarity: 0.8253)
  10. nutmeg                         (similarity: 0.8213)


## Test 4: Edge Cases

In [12]:
# Test with ingredient not in embeddings
print("Testing ingredient not found:")
try:
    engine.get_similar_ingredients("unicorn_meat")
except IngredientNotFoundError as e:
    print(f"  Caught expected error: {e}")

Testing ingredient not found:
  Caught expected error: Ingredient 'unicorn_meat' not found in embeddings. Did you mean 'corn'?


In [13]:
# Test case insensitivity
print("\nTesting case insensitivity:")
print(f"  'TOMATO' exists: {engine.ingredient_exists('TOMATO')}")
print(f"  'Tomato' exists: {engine.ingredient_exists('Tomato')}")
print(f"  'tomato' exists: {engine.ingredient_exists('tomato')}")


Testing case insensitivity:
  'TOMATO' exists: True
  'Tomato' exists: True
  'tomato' exists: True


In [14]:
# Test ingredient with spaces
print("\nTesting multi-word ingredients:")
garlic_similar = engine.get_similar_ingredients("garlic clove", top_n=5)
print("Similar to 'garlic clove':")
for i, (ing, score) in enumerate(garlic_similar, 1):
    print(f"  {i}. {ing} (similarity: {score:.4f})")


Testing multi-word ingredients:
Similar to 'garlic clove':
  1. garlic (similarity: 0.9383)
  2. red pepper (similarity: 0.8840)
  3. tomato (similarity: 0.8474)
  4. salt pepper (similarity: 0.8455)
  5. parsley (similarity: 0.8193)


## Test 5: Similarity Score Between Specific Ingredients

In [15]:
# Compare specific ingredient pairs
pairs = [
    ("tomato", "onion"),
    ("tomato", "basil"),
    ("milk", "cream"),
    ("chicken", "turkey"),
    ("tomato", "chocolate"),
]

print("Similarity scores between ingredient pairs:")
print("=" * 50)
for ing_a, ing_b in pairs:
    try:
        score = engine.get_similarity_score(ing_a, ing_b)
        print(f"  {ing_a:<15} <-> {ing_b:<15} : {score:.4f}")
    except IngredientNotFoundError as e:
        print(f"  {ing_a:<15} <-> {ing_b:<15} : NOT FOUND")

Similarity scores between ingredient pairs:
  tomato          <-> onion           : NOT FOUND
  tomato          <-> basil           : 0.7522
  milk            <-> cream           : 0.8131
  chicken         <-> turkey          : 0.7016
  tomato          <-> chocolate       : 0.4530


## Test 6: Batch Queries

In [16]:
# Batch query multiple ingredients
query_ingredients = ["tomato", "chicken", "milk"]

batch_results = engine.batch_similar_ingredients(
    query_ingredients,
    top_n=5
)

print("Batch query results:")
print("=" * 50)
for query, similar in batch_results.items():
    print(f"\n{query}:")
    for ing, score in similar:
        print(f"    - {ing} ({score:.4f})")

Batch query results:

tomato:
    - garlic clove (0.8474)
    - oregano (0.8359)
    - garlic (0.8342)
    - red pepper (0.8320)
    - plum tomato (0.8126)

chicken:
    - chicken broth (0.8146)
    - garlic (0.8064)
    - red pepper (0.8052)
    - chicken thigh (0.7994)
    - garlic clove (0.7894)

milk:
    - egg (0.8934)
    - flour (0.8645)
    - margarine (0.8567)
    - cream (0.8131)
    - powder (0.7876)


## Test 7: Visualize Top-10 Results

In [17]:
# Create a simple visualization using pandas
def visualize_similar(query: str, top_n: int = 10):
    """Create a simple bar chart visualization of similar ingredients."""
    similar = engine.get_similar_ingredients(query, top_n=top_n)
    
    df = pd.DataFrame(similar, columns=["ingredient", "similarity"])
    
    # Create ASCII bar chart
    print(f"\nTop {top_n} ingredients similar to '{query}':")
    print("=" * 60)
    
    max_bar_len = 40
    max_sim = df["similarity"].max()
    
    for _, row in df.iterrows():
        bar_len = int((row["similarity"] / max_sim) * max_bar_len)
        bar = "#" * bar_len
        print(f"  {row['ingredient']:<25} |{bar:<40}| {row['similarity']:.4f}")
    
    return df

In [18]:
# Visualize for tomato
df_tomato = visualize_similar("tomato")


Top 10 ingredients similar to 'tomato':
  garlic clove              |########################################| 0.8474
  oregano                   |####################################### | 0.8359
  garlic                    |####################################### | 0.8342
  red pepper                |####################################### | 0.8320
  plum tomato               |######################################  | 0.8126
  parsley                   |#####################################   | 0.8024
  salt pepper               |####################################    | 0.7827
  cayenne pepper            |####################################    | 0.7812
  tomato sauce              |####################################    | 0.7782
  cumin                     |####################################    | 0.7749


In [19]:
# Visualize for chicken
df_chicken = visualize_similar("chicken")


Top 10 ingredients similar to 'chicken':
  chicken broth             |########################################| 0.8146
  garlic                    |####################################### | 0.8064
  red pepper                |####################################### | 0.8052
  chicken thigh             |####################################### | 0.7994
  garlic clove              |######################################  | 0.7894
  parsley                   |###################################     | 0.7330
  tomato                    |###################################     | 0.7310
  cayenne pepper            |###################################     | 0.7309
  rice                      |###################################     | 0.7286
  mushroom                  |###################################     | 0.7140


In [20]:
# Visualize for milk
df_milk = visualize_similar("milk")


Top 10 ingredients similar to 'milk':
  egg                       |########################################| 0.8934
  flour                     |######################################  | 0.8645
  margarine                 |######################################  | 0.8567
  cream                     |####################################    | 0.8131
  powder                    |###################################     | 0.7876
  shortening                |###################################     | 0.7826
  2 milk                    |##################################      | 0.7812
  vanilla extract           |##################################      | 0.7777
  buttermilk                |##################################      | 0.7713
  soda                      |##################################      | 0.7644


## Summary

The recommendation engine is working correctly:

1. **Embedding Loading**: Successfully loads 2698 ingredients with 128-dimensional embeddings
2. **Basic Queries**: Returns sensible similar ingredients based on recipe co-occurrence
3. **Filtering**: Correctly excludes specified trigger ingredients
4. **Edge Cases**: Handles missing ingredients, case variations, and multi-word names
5. **Batch Processing**: Can process multiple queries efficiently

The engine is ready to be integrated into the Flask API for the SafePlate application.

In [21]:
# Final stats
print("\n" + "=" * 50)
print("RECOMMENDATION ENGINE STATS")
print("=" * 50)
print(f"Total ingredients: {engine.num_ingredients}")
print(f"Embedding dimension: {engine.embedding_dim}")
print(f"Memory usage (matrix): ~{engine.num_ingredients * engine.embedding_dim * 8 / 1024 / 1024:.2f} MB")


RECOMMENDATION ENGINE STATS
Total ingredients: 2698
Embedding dimension: 128
Memory usage (matrix): ~2.63 MB
